In [ ]:
import wfdb
import pandas as pd
import matplotlib.pyplot as plt

# **Load Metadata**

In [ ]:
metadata = pd.read_csv('brugada-syndrome/metadata.csv')

In [ ]:
print(metadata.head())
print(f"Total subjects: {len(metadata)}")
print(f"Brugada patients: {(metadata['brugada'] > 0).sum()}")
print(f"Healthy subjects: {(metadata['brugada'] == 0).sum()}")

# **Initial Visualization**

In [ ]:
patient_id = '188981'
record = wfdb.rdrecord(f'brugada-syndrome/files/{patient_id}/{patient_id}')

signals = record.p_signal
lead_names = record.sig_name
sampling_freq = record.fs

num_leads = len(lead_names)

plt.figure(figsize=(12, 18))

for i in range(num_leads):
    plt.subplot(num_leads, 1, i + 1)
    plt.plot(signals[:, i])
    plt.title(f'Lead {lead_names[i]}')
    plt.ylabel('mV')

plt.xlabel('Sample')
plt.tight_layout()
plt.show()

# **Filtering**

In [ ]:
from scipy import signal

def signal_cleaning(raw_data):
    b, a = signal.butter(3, 0.05, btype='highpass', fs=100)
    return signal.filtfilt(b, a, raw_data)

clean_signal = signal_cleaning(signals[:, 0])


plt.figure(figsize=(12, 4))
plt.plot(signals[0:500, 0], label='Raw')
plt.plot(clean_signal[0:1100], label='Clean', color='red')
plt.legend()
plt.show()

# **Normalization**

In [ ]:
# Normalize
normal_signal = (clean_signal - clean_signal.min()) / (clean_signal.max() - clean_signal.min())


plt.figure(figsize=(12, 4))
plt.plot(normal_signal[0:500], color='green')
plt.title('Normalized Signal (Scale 0 to 1)')
plt.ylabel('New Scale')
plt.show()

# **Peak Detection**

In [ ]:
from scipy.signal import find_peaks

# Finding Peak R
peaks, _ = find_peaks(normal_signal, distance=50, height=0.4)

plt.figure(figsize=(12, 4))
plt.plot(normal_signal[0:500])
plt.plot(peaks[peaks < 500], normal_signal[peaks[peaks < 500]], "x", color='red')
plt.title('Marking Peak Heart Rate')
plt.show()

print(f"Peak position found: {peaks[:5]}")

# **Windowing**

In [ ]:
import numpy as np

heartbeat_collection = []
left_width = 40
right_width = 60

# Cutting
for p in peaks:
    if p > left_width and p < (len(normal_signal) - right_width):
        snippet = normal_signal[p - left_width : p + right_width]
        heartbeat_collection.append(snippet)

# Convert to Array
heartbeat_collection = np.array(heartbeat_collection)


plt.figure(figsize=(5, 4))
plt.plot(heartbeat_collection[0])
plt.title('First Cut (1 beat)')
plt.xlabel('Cut Width (100 sample)')
plt.show()

# **Looping**

In [ ]:
all_heartbeats = []
all_labels = []
metadata_test = metadata

In [ ]:
for index, row in metadata_test.iterrows():
    p_id = str(row['patient_id'])
    is_brugada = row['brugada']
    
    try:
        # 1. Read
        record = wfdb.rdrecord(f'brugada-syndrome/files/{p_id}/{p_id}')
        signal_raw = record.p_signal[:, 0] # Kita pakai Lead I dulu
        
        # 2. Filtering
        clean = signal_cleaning(signal_raw) 
        
        # 3. Normalization (Scale 0-1)
        norm = (clean - clean.min()) / (clean.max() - clean.min())
        
        # 4. Peak Detection
        pks, _ = find_peaks(norm, distance=50, height=0.4)
        
        # 5. Windowing
        count_snips = 0
        for p in pks:
            if p > left_width and p < (len(norm) - right_width):
                snip = norm[p - left_width : p + right_width]
                all_heartbeats.append(snip)
                all_labels.append(is_brugada)
                count_snips += 1
        
        print(f"{p_id} Success with {count_snips} Heartbeat.")
                
    except Exception as e:
        print(f"{p_id} Fail: {e}")

In [ ]:
# Convert to Array
X_test = np.array(all_heartbeats)
y_test = np.array(all_labels)

print(f"Total heartbeat cut (X_test): {X_test.shape}")
print(f"Total label/status (y_test): {y_test.shape}")